In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os
DATA_DIR = '/content/data_dir'
os.listdir(DATA_DIR)


['p11_Gate analysis.insoleX',
 'p3_running.insoleX',
 'p8_Gate Analasys.insoleX',
 'p8_Run Analasys.insoleX',
 'p2_Running.c3d',
 'p8_Run Analasys.c3d',
 'p4_gait analysis.insoleX',
 'p6_Gait_analysis.c3d',
 'p2_Gate_walking.c3d',
 'p10_Gate walk analysis.insoleX',
 '.ipynb_checkpoints',
 'p6_Run_analysis.c3d',
 'p6_Gait_analysis.insoleX',
 'p10_Gate walk analysis.c3d',
 'p2_Running.insoleX',
 'p9_Gate Analysys.insoleX',
 'p4_gait analysis.c3d',
 'p3_gait.insoleX',
 'p11_Run Analysis.insoleX',
 'p8_Gate Analasys.c3d',
 'p2_Gate walking.insoleX',
 'p5_Gait analysis.c3d',
 'p3_gait.c3d',
 'p9_Gate Analysys.c3d',
 'p5_Run analysis.c3d',
 'p11_Run Analysis.c3d',
 'p1_Gate running.c3d',
 'p4_Run analysis.c3d',
 'p10_Run analysis.insoleX',
 'p7_Gait_analysis.insoleX',
 'p4_Run analysis.insoleX',
 'p10_Run analysis.c3d',
 'p5_Gait analysis.insoleX',
 'p1_Gate walking.insoleX',
 'p7_Gait_analysis.c3d',
 'p7_run_analysis.insoleX',
 'p1_Gate running.insoleX',
 'p11_Gate analysis.c3d',
 'p6_Run_a

In [3]:
def find_files(data_dir):
    files = []
    for fname in os.listdir(data_dir):
        if not fname.endswith('.c3d'):
            continue
        fname_lower = fname.lower()

        # Sort longer prefixes first so p10/p11 don't match p1
        for p in sorted(['p1','p2','p3','p4','p5','p6','p7','p8','p9','p10','p11'],
                        key=len, reverse=True):
            if fname_lower.startswith(p):
                subject = p
                break
        else:
            continue

        # Add 'gate' as a walk keyword (common typo for 'gait')
        if 'run' in fname_lower:
            label = 1
        elif 'walk' in fname_lower or 'gait' in fname_lower or 'gate' in fname_lower:
            label = 0
        else:
            print(f'WARNING: could not determine label for {fname}')
            continue

        files.append((subject, label, os.path.join(data_dir, fname)))
    return files
files=find_files(DATA_DIR)
print(files)

[('p2', 1, '/content/data_dir/p2_Running.c3d'), ('p8', 1, '/content/data_dir/p8_Run Analasys.c3d'), ('p6', 0, '/content/data_dir/p6_Gait_analysis.c3d'), ('p2', 0, '/content/data_dir/p2_Gate_walking.c3d'), ('p6', 1, '/content/data_dir/p6_Run_analysis.c3d'), ('p10', 0, '/content/data_dir/p10_Gate walk analysis.c3d'), ('p4', 0, '/content/data_dir/p4_gait analysis.c3d'), ('p8', 0, '/content/data_dir/p8_Gate Analasys.c3d'), ('p5', 0, '/content/data_dir/p5_Gait analysis.c3d'), ('p3', 0, '/content/data_dir/p3_gait.c3d'), ('p9', 0, '/content/data_dir/p9_Gate Analysys.c3d'), ('p5', 1, '/content/data_dir/p5_Run analysis.c3d'), ('p11', 1, '/content/data_dir/p11_Run Analysis.c3d'), ('p1', 1, '/content/data_dir/p1_Gate running.c3d'), ('p4', 1, '/content/data_dir/p4_Run analysis.c3d'), ('p10', 1, '/content/data_dir/p10_Run analysis.c3d'), ('p7', 0, '/content/data_dir/p7_Gait_analysis.c3d'), ('p11', 0, '/content/data_dir/p11_Gate analysis.c3d'), ('p7', 1, '/content/data_dir/p7_run_analysis.c3d'), ('p

In [4]:
from collections import defaultdict

subject_labels = defaultdict(list)
for subj, label, _ in files:
    subject_labels[subj].append(label)

for subj in sorted(subject_labels, key=lambda x: int(x[1:])):
    labels = subject_labels[subj]
    status = "✓" if sorted(labels) == [0, 1] else "✗ PROBLEM"
    print(f"{subj}: labels={labels} {status}")

print(f"\nTotal subjects: {len(subject_labels)}")
print(f"Total files: {len(files)}")

p1: labels=[1, 0] ✓
p2: labels=[1, 0] ✓
p3: labels=[0, 1] ✓
p4: labels=[0, 1] ✓
p5: labels=[0, 1] ✓
p6: labels=[0, 1] ✓
p7: labels=[0, 1] ✓
p8: labels=[1, 0] ✓
p9: labels=[0, 1] ✓
p10: labels=[0, 1] ✓
p11: labels=[1, 0] ✓

Total subjects: 11
Total files: 22


In [5]:
!pip install ezc3d

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 712.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 6.6 MB/s eta 0:00:00


In [7]:
import ezc3d
from scipy.signal import butter, filtfilt, iirnotch
import numpy as np
FS = 2000
WINDOW_SEC = 1.0
WINDOW_SAMPLES = int(WINDOW_SEC * FS)
STEP = int(WINDOW_SAMPLES * 0.5)  # 50% overlap
EMG_CHANNELS = list(range(10))    # drop Emg_11, Emg_12

def preprocess(signal, fs=FS):
    b, a = butter(4, [20/(fs/2), 450/(fs/2)], btype='band')
    signal = filtfilt(b, a, signal)
    b, a = iirnotch(50/(fs/2), Q=30)
    return filtfilt(b, a, signal)

def load_file(filepath, label):
    c = ezc3d.c3d(filepath)
    data = np.array(c['data']['analogs'])[0][EMG_CHANNELS]  # (10, N)
    data = np.array([preprocess(ch) for ch in data])

    # Normalize each channel to zero mean, unit variance
    data = (data - data.mean(axis=1, keepdims=True)) / (data.std(axis=1, keepdims=True) + 1e-8)

    windows, labels = [], []
    for start in range(0, data.shape[1] - WINDOW_SAMPLES, STEP):
        windows.append(data[:, start:start + WINDOW_SAMPLES])
        labels.append(label)
    return np.array(windows), np.array(labels)

# Load per subject
subject_data = {}  # {subject: (X, y)}  X shape: (N, 10, 2000)

for subj, label, filepath in files:
    X_s, y_s = load_file(filepath, label)
    if subj not in subject_data:
        subject_data[subj] = ([], [])
    subject_data[subj][0].append(X_s)
    subject_data[subj][1].append(y_s)

for subj in subject_data:
    X = np.concatenate(subject_data[subj][0])
    y = np.concatenate(subject_data[subj][1])
    subject_data[subj] = (X, y)
    walk = (y == 0).sum()
    run  = (y == 1).sum()
    print(f"{subj}: {X.shape[0]} windows  (walk={walk}, run={run})")

p2: 304 windows  (walk=150, run=154)
p8: 310 windows  (walk=149, run=161)
p6: 311 windows  (walk=154, run=157)
p10: 294 windows  (walk=145, run=149)
p4: 288 windows  (walk=128, run=160)
p5: 306 windows  (walk=149, run=157)
p3: 305 windows  (walk=148, run=157)
p9: 274 windows  (walk=146, run=128)
p11: 300 windows  (walk=142, run=158)
p1: 306 windows  (walk=146, run=160)
p7: 324 windows  (walk=163, run=161)


In [8]:
def extract_features(windows):
    """
    windows: (N, 10, 2000)
    returns: (N, 60)  — 6 features x 10 channels
    """
    feats = []
    for w in windows:
        f = []
        for ch in w:
            # Time domain
            f.append(np.mean(np.abs(ch)))                        # MAV
            f.append(np.sqrt(np.mean(ch**2)))                    # RMS
            f.append(np.var(ch))                                  # Variance
            f.append(np.sum(np.diff(np.sign(ch)) != 0))          # Zero crossing rate

            # Frequency domain
            psd = np.abs(np.fft.rfft(ch))**2
            freqs = np.fft.rfftfreq(len(ch), 1/FS)
            mask = (freqs >= 20) & (freqs <= 450)
            f.append(np.sum(psd[mask]))                           # Total power
            f.append(np.average(freqs[mask], weights=psd[mask])) # Mean frequency
        feats.append(f)
    return np.array(feats)

# Precompute features for all subjects
subject_feats = {}  # {subject: (X_feat, y)}

for subj, (X, y) in subject_data.items():
    X_feat = extract_features(X)
    subject_feats[subj] = (X_feat, y)
    print(f"{subj}: features shape {X_feat.shape}")

p2: features shape (304, 60)
p8: features shape (310, 60)
p6: features shape (311, 60)
p10: features shape (294, 60)
p4: features shape (288, 60)
p5: features shape (306, 60)
p3: features shape (305, 60)
p9: features shape (274, 60)
p11: features shape (300, 60)
p1: features shape (306, 60)
p7: features shape (324, 60)


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd

subjects = sorted(subject_feats.keys(), key=lambda x: int(x[1:]))
results = []

for test_subj in subjects:
    train_subjects = [s for s in subjects if s != test_subj]

    X_train = np.concatenate([subject_feats[s][0] for s in train_subjects])
    y_train = np.concatenate([subject_feats[s][1] for s in train_subjects])
    X_test,  y_test  = subject_feats[test_subj]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    results.append({'subject': test_subj, 'accuracy': acc, 'f1': f1})
    print(f"Left out {test_subj}: acc={acc:.3f}  f1={f1:.3f}")

df = pd.DataFrame(results)
print(f"\nLOSO Mean  acc={df.accuracy.mean():.3f} ± {df.accuracy.std():.3f}")
print(f"LOSO Mean  f1 ={df.f1.mean():.3f} ± {df.f1.std():.3f}")

Left out p1: acc=0.925  f1=0.923
Left out p2: acc=0.556  f1=0.220
Left out p3: acc=0.607  f1=0.381
Left out p4: acc=0.944  f1=0.952
Left out p5: acc=0.807  f1=0.841
Left out p6: acc=0.913  f1=0.920
Left out p7: acc=0.571  f1=0.249
Left out p8: acc=0.926  f1=0.933
Left out p9: acc=0.894  f1=0.897
Left out p10: acc=0.510  f1=0.674
Left out p11: acc=0.857  f1=0.880

LOSO Mean  acc=0.774 ± 0.174
LOSO Mean  f1 =0.715 ± 0.290


In [10]:
for subj, (X, y) in subject_data.items():
    rms = np.sqrt(np.mean(X**2, axis=(0,2)))  # per channel RMS
    print(f"{subj}: mean RMS across channels = {rms.mean():.2f}  std={rms.std():.2f}")

p2: mean RMS across channels = 1.00  std=0.00
p8: mean RMS across channels = 1.00  std=0.00
p6: mean RMS across channels = 1.00  std=0.00
p10: mean RMS across channels = 1.00  std=0.00
p4: mean RMS across channels = 1.00  std=0.00
p5: mean RMS across channels = 1.00  std=0.00
p3: mean RMS across channels = 1.00  std=0.00
p9: mean RMS across channels = 1.00  std=0.00
p11: mean RMS across channels = 1.00  std=0.00
p1: mean RMS across channels = 1.00  std=0.00
p7: mean RMS across channels = 1.00  std=0.00


In [11]:
# Train on all-but-p1, check importances
X_train = np.concatenate([subject_feats[s][0] for s in subjects if s != 'p1'])
y_train = np.concatenate([subject_feats[s][1] for s in subjects if s != 'p1'])

scaler = StandardScaler()
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(scaler.fit_transform(X_train), y_train)

channel_names = ['R.Tib.Ant.','L.Tib.Ant.','R.Rect.Fem.','L.Rect.Fem.',
                 'L.Bic.Fem.','R.Bic.Fem.','R.Glut.Med.','L.Glut.Med.',
                 'L.Gastro','R.Gastro']
feat_names = [f"{ch}_{f}" for ch in channel_names
              for f in ['MAV','RMS','Var','ZCR','Power','MeanFreq']]

importances = pd.Series(clf.feature_importances_, index=feat_names)
print(importances.sort_values(ascending=False).head(15))

R.Glut.Med._ZCR         0.116005
R.Rect.Fem._ZCR         0.074883
R.Rect.Fem._MeanFreq    0.071426
R.Gastro_ZCR            0.058958
L.Gastro_ZCR            0.038824
R.Tib.Ant._MeanFreq     0.036557
L.Glut.Med._ZCR         0.036554
R.Bic.Fem._ZCR          0.033610
L.Rect.Fem._ZCR         0.032586
L.Bic.Fem._MeanFreq     0.030342
L.Gastro_MeanFreq       0.029839
L.Tib.Ant._ZCR          0.027384
L.Glut.Med._MeanFreq    0.024743
L.Bic.Fem._ZCR          0.024212
L.Tib.Ant._MeanFreq     0.019050
dtype: float64


In [12]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

def make_loaders(X_train, y_train, X_val, y_val, batch_size=64):
    train_ds = TensorDataset(
        torch.FloatTensor(X_train),
        torch.LongTensor(y_train)
    )
    val_ds = TensorDataset(
        torch.FloatTensor(X_val),
        torch.LongTensor(y_val)
    )
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True),
            DataLoader(val_ds,   batch_size=batch_size, shuffle=False))

def train_model(model, X_train, y_train, X_val, y_val,
                epochs=50, lr=1e-3, batch_size=64):
    model = model.to(DEVICE)
    opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    train_loader, val_loader = make_loaders(X_train, y_train, X_val, y_val, batch_size)

    best_acc, best_state = 0, None

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        # Validation
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                preds.extend(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
                targets.extend(yb.numpy())
        acc = accuracy_score(targets, preds)
        if acc > best_acc:
            best_acc = acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model, best_acc

def loso_eval(model_fn, data_dict, use_raw=True, epochs=50):
    """
    model_fn: callable that returns a fresh model instance
    use_raw:  True  -> data_dict values are (X_raw, y),  shape (N, 10, 2000)
              False -> data_dict values are (X_feat, y), shape (N, 60)
    """
    subjects = sorted(data_dict.keys(), key=lambda x: int(x[1:]))
    results  = []

    for test_subj in subjects:
        train_subjs = [s for s in subjects if s != test_subj]

        X_train = np.concatenate([data_dict[s][0] for s in train_subjs])
        y_train = np.concatenate([data_dict[s][1] for s in train_subjs])
        X_test,  y_test = data_dict[test_subj]

        model, _ = train_model(model_fn(), X_train, y_train, X_test, y_test, epochs=epochs)

        model.eval()
        with torch.no_grad():
            preds = model(torch.FloatTensor(X_test).to(DEVICE)).argmax(1).cpu().numpy()

        acc = accuracy_score(y_test, preds)
        f1  = f1_score(y_test, preds)
        results.append({'subject': test_subj, 'accuracy': acc, 'f1': f1})
        print(f"Left out {test_subj}: acc={acc:.3f}  f1={f1:.3f}")

    df = pd.DataFrame(results)
    print(f"\nLOSO Mean  acc={df.accuracy.mean():.3f} ± {df.accuracy.std():.3f}")
    print(f"LOSO Mean  f1 ={df.f1.mean():.3f} ± {df.f1.std():.3f}")
    return df

Using device: cuda


In [13]:
class MLP(nn.Module):
    def __init__(self, input_dim=60, n_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes)
        )
    def forward(self, x):
        return self.net(x)

print("Running MLP LOSO...")
mlp_results = loso_eval(
    model_fn = lambda: MLP(input_dim=60),
    data_dict = subject_feats,  # uses the 60-dim handcrafted features
    epochs = 50
)

Running MLP LOSO...
Left out p1: acc=0.712  f1=0.753
Left out p2: acc=0.628  f1=0.689
Left out p3: acc=0.682  f1=0.696
Left out p4: acc=0.663  f1=0.698
Left out p5: acc=0.503  f1=0.534
Left out p6: acc=0.537  f1=0.589
Left out p7: acc=0.846  f1=0.836
Left out p8: acc=0.597  f1=0.687
Left out p9: acc=0.620  f1=0.620
Left out p10: acc=0.694  f1=0.731
Left out p11: acc=0.517  f1=0.515

LOSO Mean  acc=0.636 ± 0.100
LOSO Mean  f1 =0.668 ± 0.096


In [14]:
class CNN1D(nn.Module):
    def __init__(self, n_channels=10, n_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1 - broad filters, catch slow envelope (~50ms)
            nn.Conv1d(n_channels, 32, kernel_size=100, stride=5, padding=50),
            nn.BatchNorm1d(32), nn.ReLU(),
            nn.MaxPool1d(2),

            # Block 2 - medium filters
            nn.Conv1d(32, 64, kernel_size=50, stride=2, padding=25),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(2),

            # Block 3 - fine-grained
            nn.Conv1d(64, 128, kernel_size=10, stride=1, padding=5),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.AdaptiveAvgPool1d(16),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16, 256),
            nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

print("Running 1D CNN LOSO...")
cnn_results = loso_eval(
    model_fn = lambda: CNN1D(n_channels=10),
    data_dict = subject_data,  # raw windows (N, 10, 2000)
    epochs = 50
)

Running 1D CNN LOSO...
Left out p1: acc=0.993  f1=0.994
Left out p2: acc=0.970  f1=0.970
Left out p3: acc=0.987  f1=0.987
Left out p4: acc=0.906  f1=0.914
Left out p5: acc=0.938  f1=0.942
Left out p6: acc=0.990  f1=0.991
Left out p7: acc=0.978  f1=0.979
Left out p8: acc=0.987  f1=0.988
Left out p9: acc=0.938  f1=0.933
Left out p10: acc=0.592  f1=0.450
Left out p11: acc=0.957  f1=0.958

LOSO Mean  acc=0.931 ± 0.116
LOSO Mean  f1 =0.919 ± 0.158


In [16]:
class ResBlock1D(nn.Module):
    def __init__(self, channels, kernel_size=10):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2),
            nn.BatchNorm1d(channels), nn.ReLU(),
            nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2),
            nn.BatchNorm1d(channels)
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.conv(x)
        # Trim to input length in case padding causes off-by-one
        out = out[..., :x.shape[-1]]
        return self.relu(x + out)

class ResCNN1D(nn.Module):
    def __init__(self, n_channels=10, n_classes=2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(n_channels, 64, kernel_size=100, stride=5, padding=50),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(2),
        )
        self.res_blocks = nn.Sequential(
            ResBlock1D(64),
            nn.Conv1d(64, 128, kernel_size=10, stride=2, padding=5),
            nn.BatchNorm1d(128), nn.ReLU(),
            ResBlock1D(128),
            nn.Conv1d(128, 256, kernel_size=10, stride=2, padding=5),
            nn.BatchNorm1d(256), nn.ReLU(),
            ResBlock1D(256),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(8),
            nn.Flatten(),
            nn.Linear(256 * 8, 256),
            nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        return self.head(self.res_blocks(self.stem(x)))

print("Running Residual CNN LOSO...")
rescnn_results = loso_eval(
    model_fn = lambda: ResCNN1D(n_channels=10),
    data_dict = subject_data,
    epochs = 50
)

Running Residual CNN LOSO...
Left out p1: acc=0.997  f1=0.997
Left out p2: acc=0.954  f1=0.954
Left out p3: acc=0.993  f1=0.994
Left out p4: acc=0.910  f1=0.916
Left out p5: acc=0.931  f1=0.935
Left out p6: acc=0.981  f1=0.981
Left out p7: acc=0.972  f1=0.973
Left out p8: acc=0.990  f1=0.991
Left out p9: acc=0.945  f1=0.939
Left out p10: acc=0.673  f1=0.673
Left out p11: acc=0.963  f1=0.965

LOSO Mean  acc=0.937 ± 0.092
LOSO Mean  f1 =0.938 ± 0.092


In [17]:
class CNNGRUModel(nn.Module):
    def __init__(self, n_channels=10, n_classes=2):
        super().__init__()
        # CNN frontend - extract local features
        self.cnn = nn.Sequential(
            nn.Conv1d(n_channels, 64, kernel_size=100, stride=5, padding=50),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=20, stride=2, padding=10),
            nn.BatchNorm1d(128), nn.ReLU(),
        )
        # GRU over CNN output sequence
        self.gru = nn.GRU(
            input_size=128,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * 2, 128),  # *2 for bidirectional
            nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        x = self.cnn(x)              # (B, 128, T')
        x = x.permute(0, 2, 1)      # (B, T', 128)
        _, h = self.gru(x)           # h: (4, B, 128)
        h = torch.cat([h[-2], h[-1]], dim=-1)  # (B, 256) last layer both directions
        return self.classifier(h)

print("Running CNN-GRU LOSO...")
cnngru_results = loso_eval(
    model_fn = lambda: CNNGRUModel(n_channels=10),
    data_dict = subject_data,
    epochs = 50
)

Running CNN-GRU LOSO...
Left out p1: acc=0.993  f1=0.994
Left out p2: acc=0.944  f1=0.943
Left out p3: acc=0.990  f1=0.990
Left out p4: acc=0.844  f1=0.859
Left out p5: acc=0.935  f1=0.938
Left out p6: acc=0.965  f1=0.966
Left out p7: acc=0.966  f1=0.966
Left out p8: acc=0.987  f1=0.988
Left out p9: acc=0.942  f1=0.939
Left out p10: acc=0.660  f1=0.708
Left out p11: acc=0.950  f1=0.951

LOSO Mean  acc=0.925 ± 0.097
LOSO Mean  f1 =0.931 ± 0.083
